# Version 9 — Check the Answer: a Self-Check Feedback Loop

Version 8 made the final answer a typed `ModelDecision`. Version 9 **checks** that decision against the facts in state before the run ends. If it does not match, the agent sends the model a short feedback message and lets it correct itself, a bounded number of times.

## Workflow

```text
User question
     |
     v
NLQ agent loop (same as Version 8: lookups, validate, run_query, budget, cleaner)
     |
     v
model returns ModelDecision(entities=["Mars"], answerable=True)
     |
     v
after_model: check_decision(decision, captured_result)       <- NEW
     |
     +-- no problems ------------------------------> END -> assemble() -> NLQResponse
     |
     +-- problems, retries left --> "SELF-CHECK FAILED ... entity 'Mars' is not in the
     |                                result" --> back to the model --> new ModelDecision
     |
     +-- problems, no retries left -> END, problems recorded in result_note
```

Two kinds of model-level middleware are added:

| Middleware | Hook | What it repeats | Visible in the trace? |
|---|---|---|---|
| `model_retry()` | `wrap_model_call` | the same **call**, after a throttle / 503 | no |
| `ResponseCheckMiddleware` | `after_model` | the **answer**, after feedback | yes |

## What changes from Version 8

| | Version 8 | Version 9 |
|---|---|---|
| Final `ModelDecision` | accepted as returned | **checked** against `captured_result` |
| If it does not match the facts | passed to the supervisor | feedback message → the model corrects it (max 2 rounds) |
| Unfixed problems | — | written into `result_note` |
| Model-call failures (throttle, 503) | raise at once | retried twice, then raise |
| New state keys | — | `check_retries` (summed), `check_open_problems` |
| Graph | 4 nodes | **5 nodes**: `ResponseCheckMiddleware.after_model` is a node |
| Tools | 5 | unchanged |
| Prompt | V8 | V8 + one line: what to do with a SELF-CHECK message |

This notebook has two parts:
- **Part A — offline:** the four checks as a plain function, tested with mock data. No keys needed.
- **Part B — live:** the Version 8 agent with the checks and model retry added.

# Part A — The checks (offline)

## A1. Check facts, not opinions

Every check compares a field the **model** wrote with a fact the **tools** saved in Version 8:

```text
ModelDecision (model)                 captured_result (run_query)
---------------------                 ---------------------------
answerable = True          vs         was any result captured?
entities   = ["Mars"]      vs         do these words appear in the rows?
answerable = False         vs         were rows returned?  is there a note?
unmet_parts = []           vs         answerable = False needs something here
```

No model call, no database call. The same decision and state always give the same verdict.

In [ ]:
%pip install -q pydantic

## A2. The Version 8 schemas

These cells are unchanged from Version 8. Run them; the explanation is in Version 8.

In [ ]:
# ─────────────────────────────────────────────────────────────
# The two contracts
#
#   ModelDecision ── written by the MODEL (via ToolStrategy)
#       entities, answerable, note, unmet_parts       ← judgments only
#
#   NLQResponse ──── written by PYTHON in assemble()
#       result_shape, columns, rows, row_count, truncated, sql,  ← facts from run_query
#       entities, unmet_parts, result_note,                      ← copied from ModelDecision
#       grounding_record_ids, repair_count, usage                ← read from the message trace
#
# ModelDecision has NO rows and NO sql field on purpose: the model
# cannot write database results into the response.
# ─────────────────────────────────────────────────────────────
from typing import Any, Literal
from pydantic import BaseModel, Field


class ModelDecision(BaseModel):
    """The model's judgment about the question. No SQL, no rows."""
    entities: list[str] = Field(default_factory=list)
    answerable: bool = True
    note: str = ""
    unmet_parts: list[str] = Field(default_factory=list)


class TokenUsage(BaseModel):
    input_tokens: int = 0
    cached_input_tokens: int = 0
    output_tokens: int = 0
    total_tokens: int = 0
    llm_calls: int = 0
    model_id: str = ""


class NLQResponse(BaseModel):
    """The final payload delivered to the supervisor / chart agent."""
    result_shape: Literal["table", "empty", "unanswerable", "not_executed"]
    columns: list[str] = Field(default_factory=list)
    rows: list[list[Any]] = Field(default_factory=list)
    row_count: int = 0
    truncated: bool = False
    sql: str = ""
    entities: list[str] = Field(default_factory=list)
    unmet_parts: list[str] = Field(default_factory=list)
    grounding_record_ids: list[str] = Field(default_factory=list)
    result_note: str = ""
    repair_count: int = 0
    usage: TokenUsage = Field(default_factory=TokenUsage)


print("Schemas defined.")

## A3. `check_decision()`

Four checks, each returning a problem written as an instruction the model can act on.

In [ ]:
# ─────────────────────────────────────────────────────────────
# check_decision(decision, captured) ──► list of problems ([] = passed)
#
#   CHECK 1  answerable=True,  but no query result captured        ──► problem
#   CHECK 2  answerable=True,  an entity does not appear in the rows ──► problem
#   CHECK 3  answerable=False, rows exist, but note is empty          ──► problem
#   CHECK 4  answerable=False, but unmet_parts is empty               ──► problem
#
# Every check compares the model's JUDGMENT with FACTS already in state.
# No model call, no database call: plain Python, so it is cheap and
# gives the same answer every time.
#
# Does NOT:
#   - judge whether the SQL answered the question (that needs meaning, not facts)
#   - check numbers in the note (the note is free text)
#   - understand synonyms: CHECK 2 is a word match, "buyers" will not match "customers"
# ─────────────────────────────────────────────────────────────
import re

_WORD = re.compile(r"[A-Za-z0-9][A-Za-z0-9_\-]{2,}")   # words of 3+ characters


def _result_text(captured: dict) -> str:
    """All column names and cell values of the captured result, lower-case."""
    cells = [str(c) for c in captured.get("columns") or []]
    cells += [str(v) for row in captured.get("rows") or [] for v in row]
    return " ".join(cells).lower()


def _entity_in_result(entity: str, text: str) -> bool:
    """True if any word of 3+ characters from the entity appears in the result.

    "product 2297 (RAN-FIC-1296)" passes if "2297" or "ran-fic-1296" is in the rows.
    """
    words = [w.lower() for w in _WORD.findall(entity)]
    return not words or any(w in text for w in words)


def check_decision(decision, captured: dict | None) -> list[str]:
    problems: list[str] = []
    has_rows = bool(captured and captured.get("rows"))

    # CHECK 1 — answerable needs a result.
    if decision.answerable and not captured:
        problems.append(
            "You set answerable=true, but no query returned a result in this run. "
            "Run the query that answers the question, or set answerable=false and explain in note."
        )

    # CHECK 2 — every entity must appear in the result.
    if decision.answerable and has_rows:
        text = _result_text(captured)
        for entity in decision.entities:
            if not _entity_in_result(entity, text):
                problems.append(
                    f"Entity '{entity}' does not appear in the query result. "
                    "List only entities that appear in the result."
                )

    # CHECK 3 — "not answerable" with rows needs a reason.
    if not decision.answerable and has_rows and not decision.note.strip():
        problems.append(
            "You set answerable=false, but a query returned rows. "
            "Say in note why the result does not answer the question."
        )

    # CHECK 4 — "not answerable" must say what is missing.
    if not decision.answerable and not decision.unmet_parts:
        problems.append(
            "You set answerable=false, but unmet_parts is empty. "
            "Name the part of the question that could not be answered."
        )

    return problems

## A4. Test the checks

One passing decision and one failure for each check.

In [ ]:
# Test every check offline — no keys, no database, no model.
captured = {"columns": ["status", "count"], "rows": [["cancelled", 7]], "truncated": False}

cases = {
    "passes":                       (ModelDecision(entities=["cancelled"], note="Status = cancelled."), captured),
    "1 · answerable, no result":    (ModelDecision(entities=["cancelled"]), None),
    "2 · entity not in rows":       (ModelDecision(entities=["Mars"]), captured),
    "3 · rows but no reason":       (ModelDecision(answerable=False, unmet_parts=["trend"]), captured),
    "4 · no unmet_parts":           (ModelDecision(answerable=False, note="Not in schema."), None),
}

for label, (decision, cap) in cases.items():
    problems = check_decision(decision, cap)
    print(f"{label:30} -> {'PASS' if not problems else 'FAIL'}")
    for p in problems:
        print("      ", p)

**CHECK 2 is a word match, on purpose.** `"product 2297 (RAN-FIC-1296)"` passes if `2297` or `ran-fic-1296` is in the rows. It is strict enough to catch an invented entity and simple enough to explain. It does not understand synonyms.

# Part B — The live agent

## B1. Install and configure

| Variable | Required | Used for |
|---|---|---|
| `PINECONE_API_KEY` | yes | Reads the `ecom-kb` index |
| `PINECONE_INDEX_NAME` | no (default `ecom-kb`) | Index name |
| `PINECONE_EMBEDDING_MODEL` | no (default `text-embedding-3-small`) | Must match the model used to fill the index |
| `OPENAI_API_KEY` | yes | Query embeddings only |
| `BEDROCK_CHAT_MODEL_ID` | yes | The chat model (the brain) |
| `AWS_REGION` | no (default `us-east-1`) | Bedrock region; AWS credentials come from your normal AWS config |
| `PGHOST`, `PGDATABASE`, `PGUSER`, `PGPASSWORD` | yes | PostgreSQL connection (use a read-only role) |
| `PGPORT` | no (default `5432`) | |
| `PGSSLMODE` | no | e.g. `require` if your provider needs TLS |

Put these in a `.env` file next to the notebook. Never commit credentials.

In [ ]:
%pip install -q langchain langchain-aws langchain-openai pinecone python-dotenv "psycopg[binary]"

In [ ]:
# ─────────────────────────────────────────────────────────────
# Configuration — Version 9
#
#   .env ──► load_dotenv ──► checks ──► model        (Bedrock chat model: the brain)
#                                    ├─► embeddings   (OpenAI: turns a search phrase into a vector)
#                                    ├─► index        (Pinecone `ecom-kb`: the knowledge base)
#                                    └─► PostgreSQL settings (used by the SQL tools)
#
# The embedding model MUST be the same one used when the vectors were
# written to Pinecone, otherwise similarity scores are meaningless.
#
# Does NOT build or upload the knowledge base. It reads the existing index.
# ─────────────────────────────────────────────────────────────
import os
import json
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_aws import ChatBedrockConverse
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "ecom-kb")
EMBEDDING_MODEL = os.getenv("PINECONE_EMBEDDING_MODEL", "text-embedding-3-small")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_CHAT_MODEL_ID = os.getenv("BEDROCK_CHAT_MODEL_ID", "")

if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY.")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY for query embeddings. It must match the embedding model used to build the Pinecone index.")
if not BEDROCK_CHAT_MODEL_ID:
    raise ValueError("Set BEDROCK_CHAT_MODEL_ID to a chat model enabled in your AWS account.")

required_pg_vars = ["PGHOST", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_pg_vars = [name for name in required_pg_vars if not os.getenv(name)]
if missing_pg_vars:
    raise ValueError("Missing PostgreSQL environment variables: " + ", ".join(missing_pg_vars))

PGPORT = int(os.getenv("PGPORT", "5432"))
PGSSLMODE = os.getenv("PGSSLMODE")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pinecone = Pinecone(api_key=PINECONE_API_KEY)
index = pinecone.Index(PINECONE_INDEX_NAME)
model = ChatBedrockConverse(
    model=BEDROCK_CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

## B2. Carried over: lookups, SQL policy, `validate_query`

These cells are unchanged from Version 3 and 4. Run them; the explanation is in Version 3 and 4.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Knowledge-base tools — the agent's MEMORY
#
#   lookup_schema(term)        ──► search_namespace("nlq-schema")   ──► tables, columns, joins
#   lookup_examples(question)  ──► search_namespace("nlq-examples") ──► similar question + SQL
#   lookup_common(question)    ──► search_namespace("common")       ──► business definitions
#
#   search_namespace: text ──► embedding ──► Pinecone top-k ──► [{id, score, metadata}]
#   format_matches:   matches ──► readable text (no vectors), cut to max_chars
#
# Does NOT execute SQL and does NOT write to Pinecone.
# ─────────────────────────────────────────────────────────────


def search_namespace(namespace: str, text: str, top_k: int = 8) -> list[dict]:
    """Embed `text` and return the top-k matches from one Pinecone namespace."""
    vector = embeddings.embed_query(text)
    response = index.query(vector=vector, namespace=namespace, top_k=top_k, include_metadata=True)

    # The Pinecone client can return a dict or an object; read both shapes.
    matches = response.get("matches", []) if isinstance(response, dict) else response.matches
    results = []
    for match in matches:
        if isinstance(match, dict):
            results.append({"id": match.get("id", "unknown"),
                            "score": match.get("score"),
                            "metadata": match.get("metadata") or {}})
        else:
            results.append({"id": getattr(match, "id", "unknown"),
                            "score": getattr(match, "score", None),
                            "metadata": getattr(match, "metadata", None) or {}})
    return results


def format_matches(matches: list[dict], max_chars: int = 12000) -> str:
    """Turn matches into text blocks: 'RESULT n | id=... | score=...' plus metadata lines."""
    if not matches:
        return "No matching knowledge-base records were found."
    blocks = []
    for rank, match in enumerate(matches, start=1):
        lines = [f"RESULT {rank} | id={match['id']} | score={match['score']}"]
        for key, value in match["metadata"].items():
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            lines.append(f"{key}: {value}")
        blocks.append("\n".join(lines))
    output = "\n\n---\n\n".join(blocks)
    return output[:max_chars] + ("\n...[truncated]" if len(output) > max_chars else "")


@tool
def lookup_schema(term: str) -> str:
    """Search the nlq-schema Pinecone namespace for tables and columns.
    Use it to confirm real table/column names, data types, descriptions, synonyms,
    and business meaning. Retrieves metadata only; does not execute SQL."""
    return format_matches(search_namespace("nlq-schema", term, top_k=8))


@tool
def lookup_examples(question: str) -> str:
    """Search the nlq-examples Pinecone namespace for similar approved NL-to-SQL examples.
    Use examples as patterns, then verify schema details using lookup_schema.
    Retrieves examples only; does not execute SQL."""
    return format_matches(search_namespace("nlq-examples", question, top_k=4), max_chars=10000)


@tool
def lookup_common(question: str) -> str:
    """Search the common Pinecone namespace for shared business definitions or guidance.
    Retrieves knowledge only; does not execute SQL."""
    return format_matches(search_namespace("common", question, top_k=4), max_chars=8000)

In [ ]:
# ─────────────────────────────────────────────────────────────
# check_read_only_sql(sql) — ONE SQL policy, used by both SQL tools
#
#   sql ──► STEP 1 make a "scan copy": remove string literals, then comments
#       ──► STEP 2 exactly one statement?           (a ';' inside a string is fine)
#       ──► STEP 3 starts with SELECT or WITH?
#       ──► STEP 4 no write / admin keyword or risky function anywhere?
#       ──► (True, sql to run)  or  (False, reason)
#
# Why a scan copy: `WHERE status = 'delete requested'` must not be
# rejected because of the word inside the quotes.
#
# Does NOT talk to the database, and is NOT a complete SQL parser.
# The read-only database role is still the real boundary.
# ─────────────────────────────────────────────────────────────
import re

_FORBIDDEN = re.compile(
    r"(?is)\b("
    r"INSERT|UPDATE|DELETE|MERGE|CREATE|ALTER|DROP|TRUNCATE|GRANT|REVOKE|"
    r"CALL|COPY|VACUUM|REFRESH|LOCK|INTO|"          # INTO blocks SELECT ... INTO new_table
    r"pg_sleep|pg_read_file|pg_read_binary_file|pg_ls_dir|lo_import|lo_export|"
    r"dblink\w*|set_config|pg_terminate_backend|pg_cancel_backend"
    r")\b"
)


def _scan_copy(sql: str) -> str:
    sql = re.sub(r"'(?:[^']|'')*'", "''", sql)          # string literals -> ''
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)     # /* block comments */
    sql = re.sub(r"--[^\n]*", " ", sql)                  # -- line comments
    return sql.strip()


def check_read_only_sql(sql: str) -> tuple[bool, str]:
    """Return (True, sql_to_run) if the SQL passes the policy, else (False, reason)."""
    if not isinstance(sql, str) or not sql.strip():
        return False, "SQL must be a non-empty string."

    # STEP 1 — scan copy.
    scan = _scan_copy(sql)
    if not scan:
        return False, "SQL is empty after removing comments."

    # STEP 2 — one statement only (one trailing ';' allowed).
    scan = scan[:-1].rstrip() if scan.endswith(";") else scan
    if ";" in scan:
        return False, "Only one SQL statement is allowed."

    # STEP 3 — read-only statement types only.
    if not re.match(r"(?is)^(SELECT|WITH)\b", scan):
        return False, "Only read-only SELECT or WITH queries are allowed."

    # STEP 4 — no writes hidden anywhere (e.g. inside a CTE).
    found = _FORBIDDEN.search(scan)
    if found:
        return False, f"The SQL contains a write or administrative operation ({found.group(0).upper()}) and was rejected."

    # The SQL we run is the ORIGINAL text (literals intact), minus trailing ';'.
    return True, sql.strip().rstrip(";").strip()

In [ ]:
# ─────────────────────────────────────────────────────────────
# PostgreSQL helpers (shared by the SQL tools)
#
#   pg_connect()   ──► new connection, read-only by default, with a statement timeout
#   format_rows()  ──► columns + rows ──► plain-text table the model can read
#
# Does NOT replace a read-only database role. The role is the real
# security boundary; these settings are a second layer.
# ─────────────────────────────────────────────────────────────
MAX_ROWS = 50


def pg_connect(timeout_ms: int = 15000):
    """Open a PostgreSQL connection whose transactions are read-only."""
    kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 10,
        "row_factory": dict_row,
        "options": f"-c default_transaction_read_only=on -c statement_timeout={timeout_ms}",
    }
    if PGSSLMODE:
        kwargs["sslmode"] = PGSSLMODE
    return psycopg.connect(**kwargs)


def format_rows(columns: list[str], rows: list[dict], truncated: bool) -> str:
    """Render rows as 'col | col' lines. States clearly when rows were cut off."""
    if not rows:
        return "Query succeeded. 0 rows returned."
    header = " | ".join(columns)
    body = "\n".join(" | ".join(str(row.get(c)) for c in columns) for row in rows)
    tail = (f"({len(rows)} rows shown. More rows exist; the result was capped at {MAX_ROWS}.)"
            if truncated else f"({len(rows)} rows.)")
    return f"{header}\n{body}\n{tail}"

In [ ]:
# ─────────────────────────────────────────────────────────────
# validate_query(sql) — check the SQL BEFORE running it
#
#   sql ──► STEP 1 check_read_only_sql (policy, no database)
#       │        └─ fails ──► "VALIDATION FAILED: <reason>"
#       ──► STEP 2 PostgreSQL: EXPLAIN <sql>   (parses + plans, does not run it)
#       │        └─ fails ──► "VALIDATION FAILED: PostgreSQL could not validate ...: <error>"
#       ──► "VALIDATION PASSED ... You may now call run_query(sql)." + plan preview
#
# EXPLAIN catches unknown tables/columns, type errors and syntax errors
# without touching any business data.
#
# Does NOT execute the query and does NOT return business rows.
# ─────────────────────────────────────────────────────────────
@tool
def validate_query(sql: str) -> str:
    """Validate a generated SQL query before execution.
    Checks that it is a single read-only SELECT/WITH statement and asks PostgreSQL
    EXPLAIN to parse and plan it without running the query. Call this before run_query.
    This tool does not execute the query or return business data."""
    # STEP 1 — policy.
    ok, sql_or_reason = check_read_only_sql(sql)
    if not ok:
        return f"VALIDATION FAILED: {sql_or_reason}"

    # STEP 2 — let PostgreSQL parse and plan it.
    try:
        with pg_connect(timeout_ms=10000) as conn:
            plan = conn.execute("EXPLAIN " + sql_or_reason).fetchall()
    except Exception as exc:
        # Short, actionable message; no connection details.
        message = str(exc).splitlines()[0][:500]
        return f"VALIDATION FAILED: PostgreSQL could not validate the query: {message}"

    plan_text = "\n".join(str(row.get("QUERY PLAN", row)) for row in plan[:8])
    return (
        "VALIDATION PASSED: PostgreSQL accepted the query for planning. "
        "The query has not been executed. You may now call run_query(sql) with this exact SQL.\n"
        f"Query plan preview:\n{plan_text[:3000]}"
    )

## B3. Carried over: budget and output middleware

These cells are unchanged from Version 6 and 7. Run them; the explanation is in Version 6 and 7.

In [ ]:
# ─────────────────────────────────────────────────────────────
# SQLBudgetMiddleware — at most N run_query executions per agent run
#
#   model asks for a tool
#        │
#        ▼
#   wrap_tool_call(request, handler)
#        │
#        ├─ STEP 1 not run_query?            ──► run the tool, count nothing
#        ├─ STEP 2 sql_call_count >= N?      ──► do NOT run the tool;
#        │                                       return "SQL budget reached ..." to the model
#        └─ STEP 3 otherwise                 ──► run the tool, then add 1 to sql_call_count
#                                                (a call that returns a SQL error still counts)
#
# The counter lives in AGENT STATE (`sql_call_count`). Its reducer is
# operator.add, so each update of 1 is added to the running total.
#
# Both hooks are defined:
#   wrap_tool_call   → used by agent.invoke()  / agent.stream()
#   awrap_tool_call  → used by agent.ainvoke() / agent.astream()
# Defining only the async one makes agent.invoke() raise NotImplementedError.
#
# Does NOT:
#   - limit lookups, validation or web search (they never touch the counter)
#   - reset between turns if you add a checkpointer + thread_id
#     (state persists, so the budget then covers the whole conversation)
#   - serialize parallel calls: several run_query calls requested in the SAME
#     model turn all see the same starting count
# ─────────────────────────────────────────────────────────────
import operator
from typing import Annotated
from typing_extensions import NotRequired

from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import ToolMessage
from langgraph.types import Command

SQL_TOOL_NAME = "run_query"
MAX_SQL_CALLS = 3


class SQLBudgetState(AgentState):
    # LangGraph adds each update to the existing value (0 if missing).
    sql_call_count: NotRequired[Annotated[int, operator.add]]


class SQLBudgetMiddleware(AgentMiddleware):
    """Limit calls to the run_query SQL execution tool."""

    state_schema = SQLBudgetState

    def __init__(self, max_sql_calls: int = MAX_SQL_CALLS):
        self.max_sql_calls = max_sql_calls

    # ---- shared logic -------------------------------------------------
    def _is_sql_call(self, request) -> bool:
        return request.tool_call.get("name", "") == SQL_TOOL_NAME

    def _budget_used_up(self, request) -> bool:
        return request.state.get("sql_call_count", 0) >= self.max_sql_calls

    def _budget_message(self, request) -> ToolMessage:
        return ToolMessage(
            content=(
                f"SQL budget reached: {self.max_sql_calls} run_query calls have already been used. "
                "Do not execute another SQL query. Answer using the results you already have, "
                "or explain what remains unknown."
            ),
            tool_call_id=request.tool_call["id"],
            name=SQL_TOOL_NAME,
        )

    @staticmethod
    def _add_one(result):
        """Attach `sql_call_count: +1` to whatever the tool returned."""
        if isinstance(result, ToolMessage):
            return Command(update={"messages": [result], "sql_call_count": 1})
        if isinstance(result, Command):
            update = dict(result.update or {})
            update["sql_call_count"] = update.get("sql_call_count", 0) + 1
            return Command(graph=result.graph, update=update, resume=result.resume, goto=result.goto)
        return result

    # ---- sync hook (agent.invoke) -------------------------------------
    def wrap_tool_call(self, request, handler):
        if not self._is_sql_call(request):          # STEP 1
            return handler(request)
        if self._budget_used_up(request):           # STEP 2
            return self._budget_message(request)
        return self._add_one(handler(request))      # STEP 3

    # ---- async hook (agent.ainvoke) -----------------------------------
    async def awrap_tool_call(self, request, handler):
        if not self._is_sql_call(request):          # STEP 1
            return await handler(request)
        if self._budget_used_up(request):           # STEP 2
            return self._budget_message(request)
        return self._add_one(await handler(request))  # STEP 3

In [ ]:
# ─────────────────────────────────────────────────────────────
# ToolOutputMiddleware — clean run_query output before the model reads it
#
#   run_query result
#        │
#        ▼
#   STEP 1 find the ToolMessage (plain, or inside a Command update)
#   STEP 2 clean its text:
#            empty / not text        ──► "TOOL OUTPUT VALIDATION: run_query returned an empty or non-text result."
#            starts "SQL error:" / "REFUSED:" ──► prefix "SQL execution did not return business data."
#                                               and keep the first 1,200 chars of the error
#            longer than max chars   ──► cut, then add "[Output shortened ... do not infer missing rows.]"
#            otherwise               ──► unchanged
#   STEP 3 return the same kind of object with the cleaned text
#
# Does NOT:
#   - touch other tools' output (lookups, validation, web search)
#   - decide whether a call is allowed; that is SQLBudgetMiddleware's job
#   - change what is stored anywhere else in state
# ─────────────────────────────────────────────────────────────
MAX_TOOL_OUTPUT_CHARS = 5_000


class ToolOutputMiddleware(AgentMiddleware):
    """Validate, clarify, and limit run_query output before it reaches the model."""

    def __init__(self, max_output_chars: int = MAX_TOOL_OUTPUT_CHARS):
        self.max_output_chars = max_output_chars

    # ---- shared logic -------------------------------------------------
    def clean(self, content) -> str:
        """STEP 2 — the text rules. Public so it can be tested without an agent."""
        if not isinstance(content, str) or not content.strip():
            return "TOOL OUTPUT VALIDATION: run_query returned an empty or non-text result."
        if content.lstrip().lower().startswith(("sql error:", "refused:")):
            return "SQL execution did not return business data. " + content[:1200]
        if len(content) > self.max_output_chars:
            return (
                content[: self.max_output_chars]
                + f"\n\n[Output shortened to {self.max_output_chars} characters. "
                + "The displayed result may be incomplete; do not infer missing rows.]"
            )
        return content

    def _process(self, request, result):
        if request.tool_call.get("name", "") != SQL_TOOL_NAME:
            return result

        # STEP 1 + 3 — plain ToolMessage.
        if isinstance(result, ToolMessage):
            return result.model_copy(update={"content": self.clean(result.content)})

        # STEP 1 + 3 — Command whose update carries the ToolMessage.
        if isinstance(result, Command) and isinstance(result.update, dict):
            messages = result.update.get("messages") or []
            cleaned = [
                m.model_copy(update={"content": self.clean(m.content)}) if isinstance(m, ToolMessage) else m
                for m in messages
            ]
            update = {**result.update, "messages": cleaned}
            return Command(graph=result.graph, update=update, resume=result.resume, goto=result.goto)

        return result

    # ---- sync hook (agent.invoke) -------------------------------------
    def wrap_tool_call(self, request, handler):
        return self._process(request, handler(request))

    # ---- async hook (agent.ainvoke) -----------------------------------
    async def awrap_tool_call(self, request, handler):
        return self._process(request, await handler(request))

## B4. Carried over: `run_query` that captures rows

These cells are unchanged from Version 8. Run them; the explanation is in Version 8.

In [ ]:
# ─────────────────────────────────────────────────────────────
# run_query(sql) — V8: same rules as V7, plus it CAPTURES the result in state
#
#   sql ──► STEP 1 check_read_only_sql
#       ──► STEP 2 execute on a read-only connection
#       ──► STEP 3 return a Command:
#                  messages        = [ToolMessage(text table)]     ──► what the MODEL reads
#                  captured_result = {columns, rows, truncated}    ──► what assemble() reads
#                  executed_sql    = the SQL that ran
#
# `runtime: ToolRuntime` is injected by LangChain. The model never sees
# it in the tool schema; it gives us the tool_call_id for the ToolMessage.
#
# Values are converted to JSON-friendly types (Decimal → float,
# date/datetime → ISO text) so the supervisor can serialize them.
#
# Does NOT capture failed queries: on REFUSED / SQL error the previous
# successful result (if any) stays in state.
# ─────────────────────────────────────────────────────────────
import datetime as _dt
from decimal import Decimal

from langchain.tools import ToolRuntime


class NLQState(SQLBudgetState):
    captured_result: NotRequired[dict | None]
    executed_sql: NotRequired[str]


def _plain(value):
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, (_dt.date, _dt.datetime, _dt.time)):
        return value.isoformat()
    return value


@tool
def run_query(sql: str, runtime: ToolRuntime) -> Command:
    """Execute one read-only SELECT or WITH query against PostgreSQL.
    Call validate_query first. Returns at most 50 rows as text, or a message
    starting with "SQL error:" or "REFUSED:" that explains what went wrong."""

    def reply(text: str, extra: dict | None = None) -> Command:
        message = ToolMessage(content=text, tool_call_id=runtime.tool_call_id, name="run_query")
        return Command(update={"messages": [message], **(extra or {})})

    # STEP 1 — policy.
    ok, sql_or_reason = check_read_only_sql(sql)
    if not ok:
        return reply(f"REFUSED: {sql_or_reason}")

    # STEP 2 — execute.
    try:
        with pg_connect() as conn:
            cursor = conn.execute(sql_or_reason)
            fetched = cursor.fetchmany(MAX_ROWS + 1)
            columns = [d.name for d in cursor.description] if cursor.description else []
    except psycopg.Error as exc:
        return reply(f"SQL error: {exc}")

    # STEP 3 — text for the model, structured copy for the application.
    truncated = len(fetched) > MAX_ROWS
    rows = fetched[:MAX_ROWS]
    captured = {
        "columns": columns,
        "rows": [[_plain(row.get(c)) for c in columns] for row in rows],
        "truncated": truncated,
    }
    return reply(
        format_rows(columns, rows, truncated),
        {"captured_result": captured, "executed_sql": sql_or_reason},
    )

## B5. New: `ResponseCheckMiddleware`

```text
model node ──► after_model ──► problems? ──yes──► SELF-CHECK message + jump_to="model"
                                    │
                                    no ──► END
```

Three things are new here:

1. **`after_model`** runs after every model step. The middleware acts only on the step where the model returned its `ModelDecision`, and skips tool-calling steps.
2. **`@hook_config(can_jump_to=["model"])`** lets the hook return `"jump_to": "model"`, which sends the run back to the model instead of ending it.
3. **The feedback is a `HumanMessage`** starting `SELF-CHECK FAILED`. It sits after the decision's own `ToolMessage`, so the message order stays valid for every model provider.

In [ ]:
# ─────────────────────────────────────────────────────────────
# ResponseCheckMiddleware — check the final ModelDecision, send feedback, loop back
#
#   model node ──► after_model(state)
#                     │
#                     ├─ STEP 1 last model reply did not return a ModelDecision? ──► do nothing
#                     ├─ STEP 2 check_decision(decision, captured_result)
#                     │           └─ no problems ──► check_open_problems = []  ──► END
#                     ├─ STEP 3 retries used up? ──► keep problems in state   ──► END
#                     │                              (assemble() reports them)
#                     └─ STEP 4 add a "SELF-CHECK FAILED" message, check_retries +1,
#                               jump_to="model"  ──► the model corrects its decision
#
# Why after_model and not wrap_model_call:
#   after_model writes to STATE, so the rejected decision, the feedback and the
#   retry count are all in the message trace and in result["check_retries"].
#   A wrap_model_call retry would happen inside one model step and leave no trace.
#
# Does NOT:
#   - run on tool-calling steps (STEP 1 skips them)
#   - re-check a decision that failed to parse; ToolStrategy already sends that
#     error back and the model retries on its own
#   - loop forever: at most max_retries feedback rounds per run
#   - bypass the SQL budget: if the feedback makes the model query again,
#     SQLBudgetMiddleware still counts and limits it
# ─────────────────────────────────────────────────────────────
from langchain.agents.middleware import hook_config
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

DECISION_TOOL = "ModelDecision"
MAX_CHECK_RETRIES = 2


class ResponseCheckState(AgentState):
    check_retries: NotRequired[Annotated[int, operator.add]]   # summed, like sql_call_count
    check_open_problems: NotRequired[list[str]]                 # replaced on every check


def _decision_call(state) -> dict | None:
    """The ModelDecision tool call in the latest model reply, if there is one."""
    last_ai = next((m for m in reversed(state["messages"]) if isinstance(m, AIMessage)), None)
    if last_ai is None:
        return None
    return next((c for c in last_ai.tool_calls if c["name"] == DECISION_TOOL), None)


def _parsed_ok(state, call_id: str) -> bool:
    """True if ToolStrategy accepted the decision (no 'Error:' reply for this call)."""
    for m in reversed(state["messages"]):
        if isinstance(m, ToolMessage) and m.tool_call_id == call_id:
            return not str(m.content).startswith("Error")
    return False


class ResponseCheckMiddleware(AgentMiddleware):
    """After the model returns its ModelDecision, check it against state and loop back if needed."""

    state_schema = ResponseCheckState

    def __init__(self, max_retries: int = MAX_CHECK_RETRIES):
        self.max_retries = max_retries

    @hook_config(can_jump_to=["model"])
    def after_model(self, state, runtime):
        # STEP 1 — only the step where the model returned its decision.
        call = _decision_call(state)
        if call is None or not _parsed_ok(state, call["id"]):
            return None
        decision = state.get("structured_response")
        if decision is None:
            return None

        # STEP 2 — compare the decision with the facts.
        problems = check_decision(decision, state.get("captured_result"))
        if not problems:
            return {"check_open_problems": []}

        # STEP 3 — out of retries: stop honestly, keep the problems for assemble().
        used = state.get("check_retries", 0)
        if used >= self.max_retries:
            return {"check_open_problems": problems}

        # STEP 4 — feedback the model can act on, then go back to the model.
        feedback = (
            f"SELF-CHECK FAILED (round {used + 1} of {self.max_retries}). "
            "Your ModelDecision does not match the query result:\n"
            + "\n".join(f"- {p}" for p in problems)
            + "\nReturn a corrected ModelDecision. Run another query only if a problem above needs new data."
        )
        return {
            "messages": [HumanMessage(content=feedback)],
            "check_retries": 1,
            "check_open_problems": problems,
            "jump_to": "model",
        }

## B6. New: `model_retry()` — the other kind of model wrapping

`ModelRetryMiddleware` is built into LangChain. It uses `wrap_model_call`: it repeats the **same model call** when the provider throttles or times out. Nothing is added to state, so nothing shows in the trace.

| | `model_retry()` | `ResponseCheckMiddleware` |
|---|---|---|
| Hook | `wrap_model_call` | `after_model` |
| Repeats | the call, unchanged | the answer, with feedback |
| Triggered by | an exception (throttle, 503) | a mismatch with the facts |
| In the trace | no | yes |

In [ ]:
# ─────────────────────────────────────────────────────────────
# model_retry() — silent retries of the MODEL CALL itself (wrap_model_call)
#
#   model call ──► throttled / 503 / timeout? ──► wait, call again (max 2 times)
#                                              └─► still failing ──► raise (on_failure="error")
#
# This is the other kind of model wrapping: the same call is repeated, nothing
# is added to state, nothing appears in the trace. Right for infrastructure
# errors, wrong for checking an answer (that is ResponseCheckMiddleware's job).
#
# Does NOT retry a bad ANSWER, and does NOT let the agent continue without a
# model: if the model is really down, the run fails loudly instead of guessing.
# ─────────────────────────────────────────────────────────────
from langchain.agents.middleware import ModelRetryMiddleware


def _is_retryable(exc: Exception) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    return any(marker in text for marker in
               ("throttl", "toomanyrequests", "rate", "serviceunavailable", "timeout", "503", "429"))


def model_retry() -> ModelRetryMiddleware:
    return ModelRetryMiddleware(
        max_retries=2,
        retry_on=_is_retryable,
        initial_delay=0.5,
        backoff_factor=2.0,
        on_failure="error",
    )

## B7. Changed: reading the state and assembling the response

Same as Version 8, plus the self-check outcome: `check_retries` and any problems left unfixed.

In [ ]:
# ─────────────────────────────────────────────────────────────
# facts_from_state(state) — V8 version plus the self-check result
#
#   state["captured_result"], state["executed_sql"]              ──► rows + SQL
#   validate_query ToolMessages starting "VALIDATION FAILED"     ──► repair_count
#   lookup_* ToolMessages containing "| id=<id> |"               ──► grounding_record_ids
#   AIMessage.usage_metadata                                     ──► TokenUsage
#   state["check_retries"], state["check_open_problems"]         ──► self-check outcome  (new)
#
# Does NOT call the model or the database. It only reads the state.
# ─────────────────────────────────────────────────────────────
import re as _re
from langchain_core.messages import AIMessage

_LOOKUP_TOOLS = {"lookup_schema", "lookup_examples", "lookup_common"}
_ID_PATTERN = _re.compile(r"\| id=(\S+) \|")


def facts_from_state(state: dict) -> dict:
    messages = state["messages"]
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    # STEP 1 — how many times the model had to repair its SQL.
    repair_count = sum(
        1 for m in tool_messages
        if m.name == "validate_query" and str(m.content).startswith("VALIDATION FAILED")
    )

    # STEP 2 — which knowledge-base records the model actually saw.
    grounding_ids: list[str] = []
    for m in tool_messages:
        if m.name in _LOOKUP_TOOLS:
            for record_id in _ID_PATTERN.findall(str(m.content)):
                if record_id not in grounding_ids:
                    grounding_ids.append(record_id)

    # STEP 3 — token usage across every model call.
    usage = TokenUsage(model_id=BEDROCK_CHAT_MODEL_ID)
    for m in messages:
        if isinstance(m, AIMessage):
            usage.llm_calls += 1
            meta = getattr(m, "usage_metadata", None) or {}
            usage.input_tokens += meta.get("input_tokens", 0)
            usage.output_tokens += meta.get("output_tokens", 0)
            usage.total_tokens += meta.get("total_tokens", 0)
            usage.cached_input_tokens += (meta.get("input_token_details") or {}).get("cache_read", 0)

    return {
        "captured": state.get("captured_result"),
        "executed_sql": state.get("executed_sql", ""),
        "grounding_ids": grounding_ids,
        "repair_count": repair_count,
        "usage": usage,
        "check_retries": state.get("check_retries", 0),
        "check_problems": state.get("check_open_problems") or [],
    }

In [ ]:
# ─────────────────────────────────────────────────────────────
# assemble(...) ──► NLQResponse   (V8 version plus the self-check outcome)
#
#   decision.answerable == False ──► result_shape = "unanswerable"
#   no captured result           ──► result_shape = "not_executed"
#   captured, 0 rows             ──► result_shape = "empty"
#   captured, rows               ──► result_shape = "table"
#
#   new in V9: unresolved self-check problems are added to result_note,
#   so the supervisor sees that the decision did not pass its checks.
#
# Does NOT run SQL, call the model, or change the decision.
# ─────────────────────────────────────────────────────────────
def assemble(
    decision: ModelDecision,
    captured: dict | None,
    executed_sql: str,
    grounding_ids: list[str] | None = None,
    repair_count: int = 0,
    usage: TokenUsage | None = None,
    check_retries: int = 0,
    check_problems: list[str] | None = None,
) -> NLQResponse:
    # STEP 1 — fields every shape carries.
    common = {
        "entities": decision.entities,
        "unmet_parts": decision.unmet_parts,
        "grounding_record_ids": grounding_ids or [],
        "repair_count": repair_count,
        "sql": executed_sql or "",
        "usage": usage or TokenUsage(),
    }

    # STEP 2 — the note, plus any self-check problems that were never fixed.
    note = decision.note
    if check_problems:
        note = (note + " " if note else "") + (
            f"[Self-check not passed after {check_retries} correction round(s): "
            + " | ".join(check_problems) + "]"
        )

    # STEP 3 — pick the shape.
    if not decision.answerable:
        return NLQResponse(result_shape="unanswerable", result_note=note.strip(), **common)
    if not captured:
        return NLQResponse(result_shape="not_executed",
                           result_note=(note or "no query was executed").strip(), **common)

    rows = captured.get("rows") or []
    if captured.get("truncated"):
        note = (note + " " if note else "") + f"Result capped at {len(rows)} rows; more rows exist."

    return NLQResponse(
        result_shape="table" if rows else "empty",
        columns=captured.get("columns") or [],
        rows=rows,
        row_count=len(rows),
        truncated=bool(captured.get("truncated")),
        result_note=note.strip(),
        **common,
    )

## B8. The system prompt

The Version 8 prompt with **one added line** at the end of FINAL DECISION:

```text
If you receive a SELF-CHECK FAILED message, correct the ModelDecision as it asks
and return it again.
```

Everything else is word for word the same. The feedback message itself says what is wrong and what to do.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are the NLQ agent for an e-commerce marketplace. You answer questions about the
marketplace's internal data. Your result is passed to a supervisor and a chart agent.

AVAILABLE TOOLS
1. lookup_schema(term): searches the Pinecone `nlq-schema` namespace for table and column metadata.
2. lookup_examples(question): searches the Pinecone `nlq-examples` namespace for similar questions and SQL examples.
3. lookup_common(question): searches the Pinecone `common` namespace for shared business definitions and guidance.
4. validate_query(sql): checks the read-only SQL policy and asks PostgreSQL to validate the query using EXPLAIN. It does not execute the business query.
5. run_query(sql): executes a read-only SQL query against PostgreSQL and returns internal business data.

SQL WORKFLOW
- Understand the business question, required metric, filters, and expected level of detail.
- Use lookup_examples when a similar approved SQL example may help.
- Use lookup_schema to confirm table names, columns, relationships, metric definitions, and table grain.
- Use lookup_common when shared business definitions or guidance may help.
- Treat retrieved SQL examples as patterns; do not copy mismatched joins, filters, date ranges, or columns.
- Never invent table names, columns, joins, value codes, or business definitions.
- Before executing SQL, call validate_query(sql).
- If validation fails, correct the SQL using the returned error and validate it again.
- Call run_query(sql) only after validation succeeds.
- Inspect the query result before answering. If SQL fails, use the error and schema knowledge to correct it when possible.
- Never claim that a query succeeded unless run_query confirms it.
- The last successful run_query result is the table the application returns, so make
  sure your final query is the one that answers the question.

EVIDENCE AND SAFETY
- Never invent numbers, query results, or schema details.
- Use read-only SELECT/WITH queries only and respect the validation tool's restrictions.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, state the numerator and denominator in the note.
- If the available evidence is insufficient, mark the question as not answerable and explain why.

FINAL DECISION
When you have finished using tools, return a ModelDecision:
- answerable: false only if the schema or data cannot answer the question.
- entities: the main business entities the answer is about, as they appear in the query result.
- note: one or two sentences on the definitions used and any material caveat.
- unmet_parts: parts of the question you could not answer; empty if none.
Do not put SQL, column lists, or result rows in the ModelDecision. The application
reads those directly from run_query.
If you receive a SELF-CHECK FAILED message, correct the ModelDecision as it asks
and return it again.
"""

## B9. Create the agent

In [ ]:
# ─────────────────────────────────────────────────────────────
# The V9 agent = the V8 agent + two model-level middleware
#
#   middleware=[
#       model_retry(),              wrap_model_call  — repeat a failed CALL (silent)
#       SQLBudgetMiddleware(3),     wrap_tool_call   — V6
#       ToolOutputMiddleware(5000), wrap_tool_call   — V7
#       ResponseCheckMiddleware(2), after_model      — check the ANSWER, loop back (visible)
#   ]
#
# Model hooks and tool hooks never wrap each other: model_retry wraps model
# calls only, the budget and cleaner wrap tool calls only.
# ─────────────────────────────────────────────────────────────
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    tools=[lookup_schema, lookup_examples, lookup_common, validate_query, run_query],
    system_prompt=SYSTEM_PROMPT,
    state_schema=NLQState,
    response_format=ToolStrategy(schema=ModelDecision),
    middleware=[
        model_retry(),
        SQLBudgetMiddleware(max_sql_calls=3),
        ToolOutputMiddleware(max_output_chars=5000),
        ResponseCheckMiddleware(max_retries=MAX_CHECK_RETRIES),
    ],
    name="nlq",
)

print("graph nodes:", list(agent.get_graph().nodes))

## B10. Ask a question

Look for `🔁 SELF-CHECK → MODEL` lines in the trace. Each one is a rejected decision and the feedback that was sent. The last two printed lines show how many correction rounds were used and whether any problem was left unfixed.

In [ ]:
# ─────────────────────────────────────────────────────────────
# ask_nlq(question) ──► NLQResponse   (V8 version plus the self-check trace)
#
#   STEP 1 agent.invoke(question) ──► final state
#   STEP 2 decision = state["structured_response"]  (None ──► flagged default)
#   STEP 3 facts    = facts_from_state(state)        (now includes the self-check outcome)
#   STEP 4 assemble(decision, **facts)
#
# Does NOT read state["messages"][-1] for the answer (it is a ToolMessage).
# ─────────────────────────────────────────────────────────────
def ask_nlq(question: str) -> NLQResponse:
    # STEP 1
    state = agent.invoke({"messages": [{"role": "user", "content": question}]})

    # Short trace: tool calls, observations, and every SELF-CHECK message.
    for m in state["messages"]:
        if isinstance(m, AIMessage) and m.tool_calls:
            for call in m.tool_calls:
                print("🧠 BRAIN → TOOL:", call["name"], call["args"])
        elif isinstance(m, ToolMessage):
            print(f"🖐️ TOOL → OBSERVATION ({m.name}):", str(m.content)[:200].replace("\n", " "))
        elif isinstance(m, HumanMessage) and str(m.content).startswith("SELF-CHECK"):
            print("🔁 SELF-CHECK → MODEL:", str(m.content).replace("\n", " ")[:300])

    # STEP 2
    decision = state.get("structured_response")
    if decision is None:
        decision = ModelDecision(
            note="The model did not return a structured decision; "
                 "any table shown comes from the last successful query."
        )

    # STEP 3 + 4
    facts = facts_from_state(state)
    response = assemble(decision, **facts)
    print("\nrun_query executions used:", state.get("sql_call_count", 0))
    print("self-check correction rounds:", facts["check_retries"],
          "| unresolved problems:", len(facts["check_problems"]))
    return response


response = ask_nlq("Which products generated the most revenue last month?")
response.model_dump()

In [ ]:
# A question the schema should not answer — CHECK 4 makes sure unmet_parts is filled in.
ask_nlq("What is each customer's lifetime satisfaction score?").model_dump()

## Why `after_model` and not `wrap_model_call` for the check

Both can check the decision. Inside `wrap_model_call` you would call `handler` again with feedback added:

```python
def wrap_model_call(self, request, handler):
    response = handler(request)                       # response.structured_response is parsed
    if check_decision(response.structured_response, request.state.get("captured_result")):
        return handler(request.override(messages=[*request.messages, feedback]))
    return response
```

That works, but the first decision and the feedback **never reach state**. The trace shows only the final, corrected decision, and nothing counts the retry. `after_model` writes the feedback and `check_retries` into state, so the correction can be audited and measured.

Use `wrap_model_call` for silent, mechanical retries (`model_retry()`). Use `after_model` + `jump_to` when the model should **see** what was wrong.

## Key points

- Version 8 gave the answer a shape. Version 9 checks the shape against the facts.
- The checks are plain Python on state: cheap, repeatable, testable offline.
- `after_model` + `jump_to="model"` gives a **visible** feedback loop; `wrap_model_call` gives an **invisible** retry.
- The loop is bounded (`MAX_CHECK_RETRIES = 2`). Unfixed problems are reported, never hidden.
- The SQL budget still applies if the feedback makes the model query again.
- Checks catch contradictions, not wrong meaning. A decision can pass every check and still answer the wrong question.

**Next — Version 10:** grounding injected into every model call with `wrap_model_call`, and prompt caching so that the extra context does not cost full price on every pass.